# Part 2: BRB (Belief Rule Base) engine বনাম baseline
**Robust IoT Botnet Detection Under Incomplete Network Traffic and Cross-Device Distribution Shifts**

এই notebook **নিজে থেকেই সম্পূর্ণ** (Part 1-এর variable লাগে না)। উপর থেকে নিচে একটা একটা করে Run করো, অথবা **Runtime → Run all**।

**Part 1-এর ফলাফল থেকে নেওয়া সিদ্ধান্ত**
- Feature: Part 1-এ বাছা ৬টা feature (নিচে hardcode করা)
- Source = Device 1 (Mirai দিয়ে train), Target = Device 2
- Mirai-তে cross-device shift খুব কম ছিল, তাই shift scenario-তে **Gafgyt attack** যোগ করা হয়েছে (RF-এর F1 ~০.৭৩ ছিল)

**Test scenario (৪টা)**
| নাম | মানে |
|---|---|
| Mirai A→A | একই device, চেনা attack |
| Mirai A→B | নতুন device, চেনা attack |
| Gafgyt A→A | একই device, **নতুন attack** |
| Gafgyt A→B | নতুন device + নতুন attack |

প্রতিটা ০/১০/৩০/৫০% random missing-এ। Missing শুধু **test**-এ বসানো হয়, আর সব model একই masked data পায়, তাই তুলনা ন্যায্য।

**Model**
- RF, HGB: clean data-তে train (RF-এ median imputation, HGB নিজেই NaN সামলায়)
- RF-aug, HGB-aug: ৩০% ও ৫০% missing বসানো data দিয়ে train (শক্ত baseline, reviewer এটা চাইবে)
- **BRB-ignore**: missing feature-কে ignorance ধরে rule activation থেকে বাদ
- **BRB-prior**: missing feature-এর জায়গায় training-এর grade frequency (marginalization)

## 1. Setup

In [ ]:
import glob, os, itertools, time, warnings, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import differential_evolution
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score, precision_score, recall_score
warnings.filterwarnings("ignore")

SEED = 42
SOURCE, TARGET = 1, 2
N_PER_CLASS = 20000
SELECTED = ['H_L0.01_weight', 'H_L3_weight', 'HpHp_L0.1_weight',
            'HH_jit_L0.01_mean', 'MI_dir_L5_variance', 'HpHp_L0.01_radius']
MISSING_RATES = [0.0, 0.1, 0.3, 0.5]
N_REPEATS = 3

# BRB optimisation settings (সময় কমাতে চাইলে DE_MAXITER কমাও)
N_OPT = 1500        # optimisation-এর জন্য subsample (fit অর্ধেক, eval অর্ধেক)
DE_MAXITER = 20
DE_POP = 5

In [ ]:
!pip install -q kagglehub
import kagglehub
path = kagglehub.dataset_download("mkashifn/nbaiot-dataset")
print(path)

## 2. Data load (শুধু ৬টা feature column পড়া হয়, তাই দ্রুত)

In [ ]:
def find_files(dev, kind):
    return sorted(glob.glob(f"{path}/**/{dev}.{kind}*.csv", recursive=True))

def read(f):
    return pd.read_csv(f, usecols=SELECTED)[SELECTED].astype("float32")

def load_benign(dev, n=N_PER_CLASS, seed=SEED):
    d = pd.concat([read(f) for f in find_files(dev, "benign")], ignore_index=True)
    return d.sample(min(len(d), n), random_state=seed)

def load_attack(dev, family, n=N_PER_CLASS, seed=SEED):
    files = find_files(dev, family)
    assert files, f"device {dev}-এ {family} নেই"
    per = max(1, n // len(files))
    parts = []
    for f in files:
        d = read(f)
        parts.append(d.sample(min(len(d), per), random_state=seed))
    return pd.concat(parts, ignore_index=True)

def make_xy(ben, att):
    X = np.vstack([ben.values, att.values]).astype("float64")
    y = np.r_[np.zeros(len(ben)), np.ones(len(att))]
    return X, y

# Source: Mirai (Part 1-এর মতো একই seed ও split)
benA, mirA = load_benign(SOURCE), load_attack(SOURCE, "mirai")
XA, yA = make_xy(benA, mirA)
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(XA, yA, test_size=0.3, stratify=yA, random_state=SEED)

# Target: Mirai
benB, mirB = load_benign(TARGET), load_attack(TARGET, "mirai")
Xb, yb = make_xy(benB, mirB)

# Gafgyt (নতুন attack). Benign হিসেবে training-এ না-দেখা row ব্যবহার, leakage এড়াতে
gafA, gafB = load_attack(SOURCE, "gafgyt", 10000), load_attack(TARGET, "gafgyt", 10000)
benA_held = Xa_te[ya_te == 0]
Xga = np.vstack([benA_held, gafA.values]); yga = np.r_[np.zeros(len(benA_held)), np.ones(len(gafA))]
benB10 = benB.sample(min(len(benB), 10000), random_state=SEED)
Xgb = np.vstack([benB10.values, gafB.values]); ygb = np.r_[np.zeros(len(benB10)), np.ones(len(gafB))]

SCENARIOS = {"Mirai A->A": (Xa_te, ya_te), "Mirai A->B": (Xb, yb),
             "Gafgyt A->A": (Xga, yga), "Gafgyt A->B": (Xgb, ygb)}
print("Train:", Xa_tr.shape)
for k, (X, y) in SCENARIOS.items():
    print(f"{k:12s} rows={len(y):6d}  botnet={int(y.sum()):6d}")

## 3. BRB engine
**গঠন:** ৬ feature × ৩ grade (Low/Medium/High) = **৩⁶ = ৭২৯টা rule**। প্রতিটা rule-এর output একটা belief distribution {Normal, Botnet}।

1. **Input transformation:** feature-কে `sign(x)·log1p(|x|)` করে পাশাপাশি দুই reference value-র মাঝে linear matching degree দিয়ে belief বানানো
2. **Rule activation:** `w_k ∝ ∏ (matching degree)^δ_i` (δ = attribute weight), তারপর normalize
3. **Missing feature:** ঐ attribute অন্য সব rule-এর জন্য সমান সমর্থন দেয়, অর্থাৎ rule activation-এ কোনো পক্ষপাত আনে না (AI Mode-এর কোডে যেমন `০` গুণ করে activation শূন্য করার ভুল ছিল, সেটা এখানে নেই)
4. **Rule belief:** training data থেকে `β = (activation-ওজনে class count) / (মোট ওজন + λ)`। কম সমর্থনের rule-এর belief-এর যোগফল ১-এর কম থাকে, বাকিটা **ignorance**
5. **Rule combine:** analytical Evidential Reasoning (Wang et al. 2006 সূত্র)
6. **Output:** `P(botnet) = β_botnet + ০.৫ × ignorance`
7. **Parameter optimisation:** Differential Evolution দিয়ে reference point (১৮টা) ও attribute weight (৬টা), ০% ও ৩০% missing দুই অবস্থার Brier loss মিলিয়ে

In [ ]:
GRADES = ["Low", "Medium", "High"]

def slog(X):
    return np.sign(X) * np.log1p(np.abs(X))

class BRB:
    def __init__(self, d, lam=5.0):
        self.d, self.lam = d, lam
        self.idx = np.array(list(itertools.product(range(3), repeat=d)))   # K x d
        self.K = len(self.idx)

    def match(self, Z, refs):
        n, d = Z.shape
        M = np.empty((n, d, 3))
        for i in range(d):
            r0, r1, r2 = refs[i]
            z = np.clip(Z[:, i], r0, r2)           # NaN থাকলে NaN-ই থাকে
            t1 = (z - r0) / (r1 - r0 + 1e-9)
            t2 = (z - r1) / (r2 - r1 + 1e-9)
            lo = z <= r1
            M[:, i, 0] = np.where(lo, 1 - t1, 0)
            M[:, i, 1] = np.where(lo, t1, 1 - t2)
            M[:, i, 2] = np.where(lo, 0, t2)
        return np.clip(M, 0, 1)

    def fill(self, M, Z, mode, prior=None):
        miss = np.isnan(Z)
        if not miss.any():
            return M
        f = np.ones((1, 1, 3)) if mode == "ignore" else prior[None, :, :]
        return np.where(miss[:, :, None], f, M)

    def activation(self, M, delta):
        P = M ** delta[None, :, None]
        A = P[:, 0, :]
        for i in range(1, self.d):          # rule-এর ক্রম itertools.product-এর সাথে মেলে
            A = (A[:, :, None] * P[:, i, :][:, None, :]).reshape(len(A), -1)
        return A / (A.sum(1, keepdims=True) + 1e-12)

    def fit_beliefs(self, M, y, delta, chunk=4000):
        S = np.zeros(self.K); C1 = np.zeros(self.K)
        for s in range(0, len(y), chunk):
            A = self.activation(M[s:s + chunk], delta)
            S += A.sum(0); C1 += A.T @ y[s:s + chunk]
        return np.stack([S - C1, C1], 1) / (S[:, None] + self.lam)

    def infer(self, M, delta, beta, chunk=3000):
        s = beta.sum(1); out = []
        for st in range(0, len(M), chunk):
            A = self.activation(M[st:st + chunk], delta)
            one_m = 1 - A * s[None, :]
            f = A[:, :, None] * beta[None, :, :] + one_m[:, :, None]
            P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
            mu = 1.0 / (P.sum(1) - base + 1e-12)
            out.append(mu[:, None] * (P - base[:, None]) / (1 - mu * wprod + 1e-12)[:, None])
        b = np.vstack(out)
        unk = np.clip(1 - b.sum(1), 0, 1)
        return b[:, 1] + 0.5 * unk, b

brb = BRB(len(SELECTED))
print("Rule সংখ্যা:", brb.K)

### 3.1 ER সূত্রের যাচাই (দ্রুত sanity test)

In [ ]:
# একটাই rule পুরোপুরি active হলে output = ঐ rule-এর belief হওয়া উচিত
A = np.zeros((1, 3)); A[0, 1] = 1.0
beta = np.array([[0.2, 0.8], [0.1, 0.7], [0.5, 0.5]])
s = beta.sum(1)
one_m = 1 - A * s[None, :]
f = A[:, :, None] * beta[None] + one_m[:, :, None]
P = f.prod(1); base = one_m.prod(1); wprod = (1 - A).prod(1)
mu = 1 / (P.sum(1) - base)
print("প্রত্যাশিত [0.1, 0.7]; পাওয়া:", (mu[:, None] * (P - base[:, None]) / (1 - mu * wprod)[:, None]).round(4))

## 4. Training (Differential Evolution)

In [ ]:
d = len(SELECTED)
rng0 = np.random.default_rng(SEED)

Ztr = slog(Xa_tr)
sub = rng0.choice(len(Ztr), size=min(2 * N_OPT, len(Ztr)), replace=False)
Zf, yf = Ztr[sub[:N_OPT]], ya_tr[sub[:N_OPT]]            # belief fit
Ze, ye = Ztr[sub[N_OPT:]], ya_tr[sub[N_OPT:]]            # loss মাপার জন্য (আলাদা অর্ধেক)
Ze_miss = Ze.copy(); Ze_miss[rng0.random(Ze.shape) < 0.3] = np.nan   # ৩০% missing

def decode(p):
    q = p[:3 * d].reshape(d, 3)
    delta = p[3 * d:]
    return q, delta / delta.max()

def make_refs(Z, q):
    return np.array([np.quantile(Z[:, i], q[i]) for i in range(d)])

def objective(p):
    q, delta = decode(p)
    refs = make_refs(Zf, q)
    beta = brb.fit_beliefs(brb.match(Zf, refs), yf, delta)
    pc, _ = brb.infer(brb.match(Ze, refs), delta, beta)
    Mm = brb.fill(brb.match(Ze_miss, refs), Ze_miss, "ignore")
    pm, _ = brb.infer(Mm, delta, beta)
    return 0.5 * np.mean((pc - ye) ** 2) + 0.5 * np.mean((pm - ye) ** 2)

bounds = [(0.01, 0.35), (0.35, 0.65), (0.65, 0.99)] * d + [(0.2, 1.0)] * d
x0 = np.array([0.05, 0.5, 0.95] * d + [1.0] * d)
print("শুরুর loss (default quantile 5/50/95):", round(objective(x0), 4))

t0 = time.time(); hist = []
def cb(xk, convergence=None):
    hist.append(objective(xk))
    if len(hist) % 5 == 0:
        print(f"  iter {len(hist):3d}  best loss {hist[-1]:.4f}  ({time.time()-t0:.0f}s)")

res = differential_evolution(objective, bounds, x0=x0, maxiter=DE_MAXITER, popsize=DE_POP,
                             seed=SEED, tol=0, polish=False, callback=cb)
print(f"শেষ loss {res.fun:.4f}, সময় {time.time()-t0:.0f}s")

### 4.1 চূড়ান্ত model
Optimise করা reference point ও attribute weight দিয়ে **পুরো training data** থেকে rule belief বের করা হয়।

In [ ]:
q_best, delta_best = decode(res.x)
refs_best = make_refs(Ztr, q_best)
M_tr = brb.match(Ztr, refs_best)
beta_best = brb.fit_beliefs(M_tr, ya_tr, delta_best)
prior = M_tr.mean(0)                                  # d x 3, প্রতিটা সারির যোগফল ১

print("Attribute weight (বড় = বেশি গুরুত্বপূর্ণ):")
for n_, w_ in sorted(zip(SELECTED, delta_best), key=lambda t: -t[1]):
    print(f"  {n_:22s} {w_:.2f}")
print("\nReference value (transformed scale, Low/Medium/High):")
print(pd.DataFrame(refs_best, index=SELECTED, columns=GRADES).round(2))

def brb_predict(X, mode):
    Z = slog(X)
    M = brb.fill(brb.match(Z, refs_best), Z, mode, prior)
    p, _ = brb.infer(M, delta_best, beta_best)
    return (p >= 0.5).astype(int)

## 5. Baseline model

In [ ]:
def mask_random(X, rate, rng):
    Xm = X.copy()
    Xm[rng.random(X.shape) < rate] = np.nan
    return Xm

rng_aug = np.random.default_rng(7)
X_aug = np.vstack([Xa_tr, mask_random(Xa_tr, 0.3, rng_aug), mask_random(Xa_tr, 0.5, rng_aug)])
y_aug = np.r_[ya_tr, ya_tr, ya_tr]

def rf(): return make_pipeline(SimpleImputer(strategy="median"),
                               RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1))
def hgb(): return HistGradientBoostingClassifier(random_state=SEED)

base = {"RF": rf().fit(Xa_tr, ya_tr), "RF-aug": rf().fit(X_aug, y_aug),
        "HGB": hgb().fit(Xa_tr, ya_tr), "HGB-aug": hgb().fit(X_aug, y_aug)}

MODELS = {name: (lambda X, m=m: m.predict(X)) for name, m in base.items()}
MODELS["BRB-ignore"] = lambda X: brb_predict(X, "ignore")
MODELS["BRB-prior"] = lambda X: brb_predict(X, "prior")
print("Model:", list(MODELS))

## 6. মূল experiment (কয়েক মিনিট লাগতে পারে)

In [ ]:
rows = []; t0 = time.time()
for scen, (X, y) in SCENARIOS.items():
    for rate in MISSING_RATES:
        f1s = {m: [] for m in MODELS}
        for r in range(N_REPEATS if rate > 0 else 1):
            Xm = mask_random(X, rate, np.random.default_rng(1000 + r)) if rate > 0 else X
            for m, fn in MODELS.items():
                f1s[m].append(f1_score(y, fn(Xm)))
        for m in MODELS:
            rows.append(dict(Scenario=scen, Missing=int(rate * 100), Model=m,
                             F1=np.mean(f1s[m]), Std=np.std(f1s[m])))
    print(f"{scen} শেষ ({time.time()-t0:.0f}s)")
R = pd.DataFrame(rows)
order = list(MODELS)
table = R.pivot_table(index=["Scenario", "Missing"], columns="Model", values="F1")[order]
print(table.round(4).to_string())

## 7. Graph

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=False)
style = {"RF": "--", "RF-aug": "--", "HGB": ":", "HGB-aug": ":", "BRB-ignore": "-", "BRB-prior": "-"}
for ax, scen in zip(axes.ravel(), SCENARIOS):
    for m in order:
        dd = R[(R.Scenario == scen) & (R.Model == m)]
        ax.errorbar(dd.Missing, dd.F1, yerr=dd.Std, marker="o", ms=4, ls=style[m],
                    lw=2.5 if m.startswith("BRB") else 1.3, label=m)
    ax.set_title(scen); ax.set_xlabel("Missing rate (%)"); ax.set_ylabel("F1"); ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig("brb_vs_baselines.png", dpi=200); plt.show()

## 8. Explainability (paper-এর selling point)
একটা botnet নমুনায় ৩০% feature মুছে ফেলে দেখা: কোন rule কতটা ওজনে alert দিল।

In [ ]:
def explain(x, mode="ignore", top=3):
    z = slog(x[None, :]); miss = np.isnan(z[0])
    M = brb.fill(brb.match(z, refs_best), z, mode, prior)
    A = brb.activation(M, delta_best)[0]
    p, _ = brb.infer(M, delta_best, beta_best)
    print(f"P(botnet) = {p[0]:.3f}")
    print("Missing feature:", [SELECTED[i] for i in np.where(miss)[0]] or "নেই")
    # missing attribute-এর grade ভেদে আলাদা rule গুলোকে এক করে দেখানো (ওজন-গড় belief)
    vis = [i for i in range(d) if not miss[i]]
    groups = {}
    for k in np.where(A > 1e-6)[0]:
        key = tuple(brb.idx[k, i] for i in vis)
        g = groups.setdefault(key, [0.0, np.zeros(2)])
        g[0] += A[k]; g[1] += A[k] * beta_best[k]
    for key, (w, wb) in sorted(groups.items(), key=lambda kv: -kv[1][0])[:top]:
        cond = " AND ".join(f"{SELECTED[i]} is {GRADES[g]}" for i, g in zip(vis, key))
        b = wb / w
        print(f"  IF {cond}")
        print(f"    THEN Normal={b[0]:.2f}, Botnet={b[1]:.2f}   (মোট rule weight {w:.3f})")

rng_e = np.random.default_rng(3)
i_bot = rng_e.choice(np.where(yb == 1)[0])
x = Xb[i_bot].copy(); x[rng_e.random(d) < 0.3] = np.nan
explain(x)

## 9. Result save

In [ ]:
os.makedirs("results", exist_ok=True)
R.to_csv("results/brb_vs_baselines.csv", index=False)
table.round(4).to_csv("results/brb_table.csv")
print("সংরক্ষিত: results/brb_vs_baselines.csv, results/brb_table.csv, brb_vs_baselines.png")

---
## এখন কী?
শেষের **table (Cell 6)**, **graph**, আর **explainability output (Cell 8)** আমাকে পাঠাও। BRB কোথায় জিতছে বা হারছে, তা দেখে paper-এর claim ঠিক করব।
সবচেয়ে সহজ উপায়: Colab-এ **File → Download → .ipynb** করে attach করা।